### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Kamran\AppData\Local\Temp\ipykernel_11944\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\Langchain\agentic_langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of documents with metadata."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}...")
        try:
            # Load the PDF file
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name  # Add the source file name to metadata
                doc.metadata["file_type"] = 'pdf'  # Add the file type to metadata

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages.")
            
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [4]:
# Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs("../data/pdf_files/")

Found 2 PDF files in ../data/pdf_files/

Processing ai.pdf...
Loaded 9 pages.

Processing quantum.pdf...
Loaded 10 pages.

Total documents loaded: 19


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

In [6]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 500 characters
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks = split_documents(all_pdf_documents)
chunks

Split 19 documents into 32 chunks.

Example chunk:
Content: The AI Revolution: Unlocking
Business Value and Strategic
Advantage
A  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and
Challenges
PREPARED BY:  InsightSwarm Intelli...
Metadata: {'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

### Embedding and Vector Store

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager

        Args:
            model_name : HuggingFace model name for sentence embeddings 
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading embedding model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts : List of strings to embed
        
        Returns:
            np.ndarray : Array of embeddings
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
            embeddings = np.asarray(embeddings)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return np.array([])

In [10]:
# Initialize the EmbeddingManager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2481.11it/s]


Model Loaded successfully. Embedding dimension: 384


### Vector Store

In [11]:
class VectorStore:
    """Manages a vector store for document embeddings using ChromaDB"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the VectorStore

        Args:
            collection_name : Name of the ChromaDB collection
            persist_directory : Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistence ChromaDB client
            print(f"Initializing ChromaDB client...")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "Collection for PDF document embeddings"}
                )
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents : List of document metadata dictionaries
            embeddings : Corresponding embeddings as a numpy array
        """

        if not self.collection:
            raise ValueError("ChromaDB collection is not initialized.")

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        # Prepare data for insertion
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = [] 

        print(f"Adding {len(documents)} documents to the vector store...")

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)  # Copy metadata
            metadata['doc_index'] = i  # Add index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)

            # Document Content
            documents_texts.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

In [12]:
vector_store = VectorStore()
vector_store

Initializing ChromaDB client...
ChromaDB collection 'pdf_documents' initialized successfully.
Existing documents in collection: 32


In [13]:
chunks

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026'),
 Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf_files\\ai.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2', 'source_file': 'ai.pdf', 'file_type': 'pdf'}, page_content='Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................

In [14]:
# Convert text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['The AI Revolution: Unlocking\nBusiness Value and Strategic\nAdvantage\nA  Comprehensive  Analysis  of  Artificial  Intelligence  Trends,  Opportunities,  and\nChallenges\nPREPARED BY:  InsightSwarm Intelligence Agent\nDATE OF ISSUE:  August 2026',
 'Table of Contents\nMARKET SIZE\n145.2B\n+12.4% CAGR\n(2020-2026)\nADOPTION RATE\n78%\nAcross Fortune 500\nCompanies\nFUNDING LEVEL\n18.4B\nTotal Venture Capital\nInflow\nENTERPRISE USERS\n4.2M\nActive Deployments\nglobally\n...............................................................................................................................................................\n...........................................................1. Introduction & Context\n...............................................................................................................................................................\n...........................................................2. Market Landscape & Analysis\n.........................

In [15]:
# Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 32 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.00s/it]

Generated embeddings with shape: (32, 384)
Adding 32 documents to the vector store...
Successfully added 32 documents to the vector store.
Total documents in collection after addition: 64


### Retriever Pipeline

In [16]:
class RAGRetriever:
    """Retrieves relevant documents from the vector store based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAGRetriever

        Args:
            vector_store : Instance of VectorStore
            embedding_manager : Instance of EmbeddingManager
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve top_k relevant documents for the given query.

        Args:
            query : The input query string
            top_k : Number of top documents to retrieve
            score_threshold : Minimum similarity score for retrieved documents

        Returns:
            List of dictionaries containing document content and metadata
        """

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i ,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents for query: '{query}'")
            else:
                print(f"No documents found for query: '{query}'")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents for query '{query}': {e}")
            return []

In [18]:
# Retriever
rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever

In [19]:
rag_retriever.retrieve("What is Quantum Computing?")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents for query: 'What is Quantum Computing?'


[{'id': 'doc_13017f72_31',
  'content': 'References\n"Quantum Computing: A New Paradigm for Computing" by IBM Research\n"Quantum Computing: A Guide to the Technology and Its Applications" by Google Quantum\nAI Lab\n"Quantum Computing: A Review of the Current State and Future Prospects" by Microsoft\nQuantum\n"Quantum Computing: A New Era for Computing" by Nature\n"Quantum Computing: A Guide to the Future of Computing" by Scientific American\n1. \n2. \n3. \n4. \n5. \nInsightSwarm Agent Ecosystem Page 10 of 10',
  'metadata': {'page_label': '10',
   'file_type': 'pdf',
   'doc_index': 31,
   'creationdate': '',
   'producer': 'WeasyPrint 69.0',
   'content_length': 459,
   'source_file': 'quantum.pdf',
   'creator': 'PyPDF',
   'page': 9,
   'source': '..\\data\\pdf_files\\quantum.pdf',
   'total_pages': 10},
  'similarity_score': 0.4383866786956787,
  'distance': 0.5616133213043213,
  'rank': 1},
 {'id': 'doc_0afef6aa_31',
  'content': 'References\n"Quantum Computing: A New Paradigm for

In [20]:
rag_retriever.retrieve("What are the best practices and recommendations for using Quantum Computing?")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents for query: 'What are the best practices and recommendations for using Quantum Computing?'


[{'id': 'doc_c23b1db3_28',
  'content': "Business Impact\nIBM Quantum's quantum computing platform has had a significant impact on the business, with\nseveral companies already investing in the technology.\n5. Best Practices & Tactical Recommendations\nBest Practices\nDevelop a scalable and controllable quantum computing architecture\nInvest in quantum computing research and development\nDevelop quantum algorithms and software\nTactical Recommendations\nInvest in quantum computing hardware and software products\nDevelop a cloud-based quantum computing platform\nAddress quantum noise and error correction\n6. Future Trends & Strategic Outlook\nThe quantum computing market is expected to grow significantly in the coming years. Several\ncompanies are already investing heavily in the field, and there are several startups and research\ninstitutions working on developing quantum computing technology.\nFuture Trends\nQuantum computing hardware and software products will become more advanced an

### Integration VectorDB Context Pipeline with LLM output

In [24]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Load the Groq API key from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize the ChatGroq model
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

In [34]:
def rag_simple(query, retriever, llm, top_k=1):
    # Retrieve only the single strongest relevant chunk
    results = retriever.retrieve(query, top_k=top_k)
    #print(results)

    if not results:
        return "No relevant context found for the query."

    best_result = max(results, key=lambda doc: doc["similarity_score"])
    context = best_result["content"]

    # Generate one best answer using the strongest context
    prompt = f"""Use the following context to answer the question with one best answer.
    If the context does not contain the answer, respond with "I don't know."

    Context:
    {context}

    Question: {query}
    Answer:
    """
    response = llm.invoke([prompt])

    return response.content.strip()

In [40]:
answer = rag_simple("Give the structure of the ai document", rag_retriever, llm)
print(f"Answer: {answer}")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents for query: 'Give the structure of the ai document'


Answer: The structure of the AI document appears to be a table or chart with the following columns and rows:

- Column 1: AI Market Share
- Column 2: Percentage
- Column 3: Value in parentheses

The rows are:

1. Machine Learning
2. Natural Language Processing
3. Computer Vision


### Enhanced RAG Pipeline Features

In [41]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence scores, and optionally the context used for answering.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found for the query.",
            "sources": [],
            "confidence": 0.0,
            "context": ''
        }

    # Prepare context and sources
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [{
        'source' : doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
        'page' : doc["metadata"].get("page", "unknown"),
        'score' : doc["similarity_score"],
        'preview' : doc["content"][:200] + "...", # First 200 characters of the content
    } for doc in results]
    confidece = max(doc["similarity_score"] for doc in results)

    # Generate answer using the context
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        "answer" : response.content,
        "sources" : sources,
        "confidence" : confidece
    }
    if return_context:
        output["context"] = context
    return output

In [42]:
# Example usage of the advanced RAG pipeline
result = rag_advanced("What are the best practices and recommendations for using Quantum Computing?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence: {result['confidence']}")
print(f"Context preview: {result['context'][:200]}...")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents for query: 'What are the best practices and recommendations for using Quantum Computing?'


Answer: The best practices and recommendations for using Quantum Computing are:

**Best Practices:**

1. Develop a scalable and controllable quantum computing architecture.
2. Invest in quantum computing research and development.
3. Develop quantum algorithms and software.

**Tactical Recommendations:**

1. Invest in quantum computing hardware and software products.
2. Develop a cloud-based quantum computing platform.
3. Address quantum noise and error correction.
Sources: [{'source': 'quantum.pdf', 'page': 7, 'score': 0.27027428150177, 'preview': "Business Impact\nIBM Quantum's quantum computing platform has had a significant impact on the business, with\nseveral companies already investing in the technology.\n5. Best Practices & Tactical Recommen..."}, {'source': 'quantum.pdf', 'page': 7, 'score': 0.27027428150177, 'preview': "Business Impact\nIBM Quantum's quantum computing platform has had a significant impact on the business, with\nseveral companies already investing in the techno

### Advanced RAG Pipeline

In [43]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

In [47]:
# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what are the challenges in AI market?", top_k=3, min_score=0.1, stream=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents for query: 'what are the challenges in AI market?'
Streaming answer:
Use the following context to answer the question concisely.
Context:
2. Market Landscape & Analysis
The  AI  market  is  diverse,  with  various  segments,  including  machine  learning,  natural  language
processing, and computer vision.
Background
The AI market is expected to reach 145.2 billion by 2026, growing at a

 CAGR of 12.4%. This growth is
driven  by  increasing  demand  for  AI-powered  solutions,  advancements  in  technology,  and
decreasing costs.
Detailed Explanation
The AI market can be segmented into various categories, including:
Machine Learning: A subset of AI that enables systems to learn from data and improve their
performance over time.
• 
InsightSwarm Agent Ecosystem Page 6 of 9

2. Market Landscape & Analysis
The  AI  market  is  diverse,  with  various  segments,  including  machine  learning,  natural  language
processing, and computer vision.
Background
The AI market is expected to reach 145.2 billion by 2026, growing at a CAGR of 12.4%. This growth is
driven  by  increasing  demand  for  AI-powered  solutions,  advancements  in  technology,  and
decreasing costs.
Detailed Explanation
The AI market can be segmented into various categories, including:
Machine Learning: A subset of AI that enables systems to learn from data and improve their
performance over time.
• 
Insight